# Philippine Bill Detection by Color (DSP)
The color of each bill identifies its denomination:

| Color | Bill |
|---|---|
| Orange | 20 Peso |
| Red | 50 Peso |
| Violet | 100 Peso |
| Green | 200 Peso |
| Yellow | 500 Peso |
| Blue | 1000 Peso |

Signal-processing pipeline:

1. **Low-pass filter** (Gaussian) to suppress camera noise
2. **Hue classification**: every pixel is assigned to the bill color with the closest hue
3. **Median filter** on each color map to remove speckle, then **morphological closing** to bridge the printed designs
4. **Contour analysis** to find the bills, remove duplicates and measure them

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMAGE_PATH = 'Images/bills.png'   # change this to your image
MAX_SIDE = 900                    # large photos are shrunk to this size (aspect ratio kept)

img = cv2.imread(IMAGE_PATH)
if img is None:
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

scale = MAX_SIDE / max(img.shape[:2])
if scale < 1:
    img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)

rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
print("Working size:", (h, w))

## 1. Low-pass filtering and HSV
A Gaussian blur averages neighboring pixels, which removes noise. HSV separates the color (hue) from the brightness, so lighting changes affect the hue much less than they affect RGB. In OpenCV, hue runs from 0 to 179.

In [ ]:
SIGMA = 0.003 * max(h, w)
smooth = cv2.GaussianBlur(img, (0, 0), SIGMA)
hsv = cv2.cvtColor(smooth, cv2.COLOR_BGR2HSV)
hue, sat, val = [hsv[..., i].astype(np.int16) for i in range(3)]

fig, axs = plt.subplots(1, 3, figsize=(16, 5))
axs[0].imshow(rgb);                axs[0].set_title("Original")
axs[1].imshow(hue, cmap='hsv');    axs[1].set_title("Hue")
axs[2].imshow(sat, cmap='gray');   axs[2].set_title("Saturation")
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

## 2. Classifying every pixel by hue
Each denomination has a reference hue. A pixel is given to the color whose hue is closest (hue is circular, so 178 and 2 are close, which matters for red). A pixel only counts if it is colorful and bright enough, which keeps white paper, gray and dark areas out.

If a bill is missed or mislabeled, the table at the end shows its measured hue, saturation and brightness. Use those numbers to adjust `hue`, `s_min` and `v_min` here.

In [ ]:
# name, peso value, reference hue (0-179), minimum saturation, minimum brightness, drawing color (RGB)
DENOMS = [
    {"name": "Orange", "value": 20,   "hue": 14,  "s_min": 100, "v_min": 100, "color": (255, 140,   0)},
    {"name": "Red",    "value": 50,   "hue": 178, "s_min": 100, "v_min": 100, "color": (255,   0,   0)},
    {"name": "Violet", "value": 100,  "hue": 150, "s_min": 35, "v_min": 40, "color": (148,   0, 211)},
    {"name": "Green",  "value": 200,  "hue": 50,  "s_min": 35, "v_min": 40, "color": (  0, 150,   0)},
    {"name": "Yellow", "value": 500,  "hue": 23,  "s_min": 100, "v_min": 100, "color": (230, 190,   0)},
    {"name": "Blue",   "value": 1000, "hue": 103, "s_min": 55, "v_min": 45, "color": (  0,   0, 255)},
]
HUE_TOL = 12      # a pixel must be within this many hue steps of a reference hue

def hue_gap(a, b):
    d = np.abs(a - b)
    return np.minimum(d, 180 - d)          # circular distance

gap = np.full((len(DENOMS), h, w), 99, dtype=np.int16)
for k, d in enumerate(DENOMS):
    allowed = (sat >= d["s_min"]) & (val >= d["v_min"])
    gap[k] = np.where(allowed, hue_gap(hue, d["hue"]), 99)

best = gap.argmin(axis=0)
class_map = np.where(gap.min(axis=0) <= HUE_TOL, best, -1)    # -1 = not a bill color

# show the class map in the drawing colors
class_img = np.full((h, w, 3), 255, dtype=np.uint8)
for k, d in enumerate(DENOMS):
    class_img[class_map == k] = d["color"]

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].imshow(rgb);       axs[0].set_title("Original")
axs[1].imshow(class_img); axs[1].set_title("Pixels classified by hue")
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

## 3. Finding the bills
For each color:

- a **median filter** (a majority vote over a small window) removes isolated wrong pixels,
- a **morphological closing** fills the gaps left by white portraits and printed text,
- **contours** give one blob per bill, and tiny blobs are ignored.

Two colors can claim the same bill when its design contains a second hue. The larger blob is kept and the smaller one covered by it is dropped.

In [ ]:
K = int(0.05 * min(h, w)) | 1                        # closing kernel size (odd)
close_k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (K, K))
MIN_AREA = 0.012 * h * w                             # blobs smaller than 1.2% of the image are ignored

def median_hue(pixels, ref):
    # average hue measured around the reference, so red (which wraps around 0/179) is handled
    signed = (hue[pixels] - ref + 90) % 180 - 90
    return (ref + np.median(signed)) % 180

candidates = []
for k, d in enumerate(DENOMS):
    m = ((class_map == k) * 255).astype(np.uint8)
    m = cv2.medianBlur(m, 9)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, close_k)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in contours:
        area = cv2.contourArea(c)
        if area < MIN_AREA:
            continue
        blob = np.zeros((h, w), np.uint8)
        cv2.drawContours(blob, [c], -1, 255, -1)
        pixels = (blob > 0) & (class_map == k)
        if not pixels.any():
            continue
        candidates.append({
            "k": k, "contour": c, "area": area, "box": cv2.boundingRect(c),
            "H": median_hue(pixels, d["hue"]),
            "S": float(np.median(sat[pixels])),
            "V": float(np.median(val[pixels])),
        })

# remove duplicates: keep the biggest blob, drop any smaller blob mostly inside it
candidates.sort(key=lambda c: -c["area"])
bills = []
for c in candidates:
    x, y, bw, bh = c["box"]
    covered = False
    for o in bills:
        ox, oy, ow, oh = o["box"]
        ix = max(0, min(x + bw, ox + ow) - max(x, ox))
        iy = max(0, min(y + bh, oy + oh) - max(y, oy))
        if ix * iy > 0.5 * bw * bh:
            covered = True
            break
    if not covered:
        bills.append(c)

bills.sort(key=lambda c: (DENOMS[c["k"]]["value"], c["box"][0]))
print(f"Bills found: {len(bills)}")

## 4. Result

In [ ]:
out = rgb.copy()
for c in bills:
    d = DENOMS[c["k"]]
    box = cv2.boxPoints(cv2.minAreaRect(c["contour"])).astype(np.int32)
    cv2.polylines(out, [box], True, d["color"], 3)

    text = f'{d["name"]} - {d["value"]} Peso'
    x, y = c["box"][0], c["box"][1]
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    ty = max(y - 10, th + 8)
    cv2.rectangle(out, (x, ty - th - 6), (x + tw + 8, ty + 6), (255, 255, 255), -1)
    cv2.putText(out, text, (x + 4, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.6, d["color"], 2)

plt.figure(figsize=(11, 8))
plt.imshow(out); plt.axis("off"); plt.title("Detected Bills")
plt.show()

# measured colors, useful for tuning DENOMS
print(f"{'Bill':<8}{'Hue':>6}{'Sat':>6}{'Val':>6}   (reference hue)")
for c in bills:
    d = DENOMS[c["k"]]
    print(f"{d['name']:<8}{c['H']:6.0f}{c['S']:6.0f}{c['V']:6.0f}   ({d['hue']})")
print()

print("Detected bills:")
total_value = 0
for c in bills:
    d = DENOMS[c["k"]]
    print(f"  {d['name']}: {d['value']} Peso")
    total_value += d["value"]

print(f"\nTotal count: {len(bills)}")
print(f"Total value: P{total_value}")